In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-assignment-3"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
user_address_schema = StructType([
    StructField("city",StringType()),
    StructField("street",StringType()),
    StructField("state",StringType()),
    StructField("postal_code",StringType()),
])
user_schema = StructType ([
    StructField("user_id",LongType()),
    StructField("user_first_name",StringType()),
    StructField("user_last_name",StringType()),
    StructField("user_email",StringType()),
    StructField("user_gender",StringType()),
    StructField("user_phone_numbers",ArrayType(StringType())),
    StructField("user_address",user_address_schema)    
])

In [3]:
df1 = spark.read.format('json').schema(user_schema).load('/public/sms/users')

In [4]:
df1.show(5)

+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+
|user_id|user_first_name|user_last_name|          user_email|user_gender|  user_phone_numbers|        user_address|
+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+
| 200001|         Eirena|     Cutsforth|ecutsforth0@wisc.edu|     Female|[4197404036, 9173...|{Dallas, 8 Warrio...|
| 200002|          Marja|      Shopcott|mshopcott1@hexun.com|     Female|[9542037028, 2128...|{Joliet, 66 Prair...|
| 200003|           Dawn|       Tointon|  dtointon2@ucsd.edu|     Female|[9523035647, 2134...|{Shawnee Mission,...|
| 200004|          Goldi|        Leaman|     gleaman3@360.cn|     Female|[2027069459, 7042...|{Saint Paul, 7696...|
| 200005|       Brewster|      Hallagan|bhallagan4@livejo...|       Male|[8134746319, 2152...|{Albuquerque, 942...|
+-------+---------------+--------------+--------------------+-----------

In [5]:
#{"user_id":1,
# "user_first_name":"Lezley",
# "user_last_name":"D'Alessio",
# "user_email":"ldalessio0@google.com.au",
# "user_gender":"Male",
# "user_phone_numbers":["5639521582","8433335556","9193704732","8326122969"],
# "user_address":{"street":"28470 Di Loreto Point","city":"Albany","state":"New York","postal_code":"12222"}}

In [26]:
df2 = df1.filter("size(user_phone_numbers) >0" ).withColumn("user_state",col("user_address.state"))

In [28]:
df2.show(3)

+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+----------+
|user_id|user_first_name|user_last_name|          user_email|user_gender|  user_phone_numbers|        user_address|user_state|
+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+----------+
| 200001|         Eirena|     Cutsforth|ecutsforth0@wisc.edu|     Female|[4197404036, 9173...|{Dallas, 8 Warrio...|     Texas|
| 200002|          Marja|      Shopcott|mshopcott1@hexun.com|     Female|[9542037028, 2128...|{Joliet, 66 Prair...|  Illinois|
| 200003|           Dawn|       Tointon|  dtointon2@ucsd.edu|     Female|[9523035647, 2134...|{Shawnee Mission,...|    Kansas|
+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+----------+
only showing top 3 rows



In [35]:
df2.groupBy("user_state").pivot("user_gender").count().orderBy("user_state").show()

+--------------------+------+-----+
|          user_state|Female| Male|
+--------------------+------+-----+
|             Alabama|  9178| 9307|
|              Alaska|  1938| 1882|
|             Arizona|  9543| 9406|
|            Arkansas|  2416| 2420|
|          California| 48716|49120|
|            Colorado| 10125|10128|
|         Connecticut|  5917| 5797|
|            Delaware|  1654| 1651|
|District of Columbia| 14292|14212|
|             Florida| 36688|36692|
|             Georgia| 13028|13008|
|              Hawaii|  2062| 2172|
|               Idaho|  2101| 2058|
|            Illinois| 11267|11178|
|             Indiana|  9676| 9604|
|                Iowa|  4726| 4706|
|              Kansas|  5776| 5962|
|            Kentucky|  6108| 6216|
|           Louisiana|  8631| 8706|
|               Maine|   228|  225|
+--------------------+------+-----+
only showing top 20 rows



In [30]:
df2.createOrReplaceTempView("users")

In [34]:
spark.sql("""
select * from (
select user_state, user_gender from users 
) 
PIVOT ( count(*) for user_gender in ('Male','Female')
) order by user_state
""").show()

+--------------------+-----+------+
|          user_state| Male|Female|
+--------------------+-----+------+
|             Alabama| 9307|  9178|
|              Alaska| 1882|  1938|
|             Arizona| 9406|  9543|
|            Arkansas| 2420|  2416|
|          California|49120| 48716|
|            Colorado|10128| 10125|
|         Connecticut| 5797|  5917|
|            Delaware| 1651|  1654|
|District of Columbia|14212| 14292|
|             Florida|36692| 36688|
|             Georgia|13008| 13028|
|              Hawaii| 2172|  2062|
|               Idaho| 2058|  2101|
|            Illinois|11178| 11267|
|             Indiana| 9604|  9676|
|                Iowa| 4706|  4726|
|              Kansas| 5962|  5776|
|            Kentucky| 6216|  6108|
|           Louisiana| 8706|  8631|
|               Maine|  225|   228|
+--------------------+-----+------+
only showing top 20 rows

